# Regressão de Potência — Otimização Bayesiana (Optuna)
**Projeto Final — CKP8277 Aprendizagem Automática · UFC**

**Aluno:** Diego Melo — 603127

Alternativa ao RandomizedSearchCV usando **Optuna** com o sampler TPE (Tree-structured Parzen Estimator).  
O TPE aprende com os trials anteriores e direciona a busca para regiões promissoras do espaço de hiperparâmetros.

Modelos:
- Random Forest Regressor
- XGBoost Regressor
- LightGBM Regressor
- CatBoost Regressor
- KNN Regressor
- SARIMA (baseline temporal)

---
## 0. Instalação de Dependências

Execute apenas uma vez.

In [ ]:
%pip install -q \
    numpy pandas matplotlib seaborn scipy scikit-learn \
    xgboost lightgbm catboost pmdarima joblib optuna \
    "mlflow>=2.14,<3" "protobuf<5" openpyxl statsmodels

---
## 1. Google Drive (Colab)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## 2. Configuração Global

In [ ]:
import warnings, sys, os, json, joblib
warnings.filterwarnings('ignore')
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)  # silencia logs internos

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from pmdarima import auto_arima

import mlflow
import mlflow.sklearn

# ── GPU ───────────────────────────────────────────────────────────────────────
def detect_gpu():
    try:
        import subprocess
        return subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
    except: return False

GPU_AVAILABLE = detect_gpu()
XGB_DEVICE    = "cuda" if GPU_AVAILABLE else "cpu"
LGBM_DEVICE   = "gpu"  if GPU_AVAILABLE else "cpu"
CAT_TASK_TYPE = "GPU"  if GPU_AVAILABLE else "CPU"

# ── Tema visual ───────────────────────────────────────────────────────────────
PALETTE = "Set2"; PRIMARY = "#2D7DD2"; ACCENT = "#F18F01"; NEUTRAL = "#6C757D"
sns.set_theme(style="whitegrid", palette=PALETTE)
DPI, FIG_W, FIG_H = 120, 14, 4.5

# ── Experimento ───────────────────────────────────────────────────────────────
RANDOM_STATE = 42
N_TRIALS     = 100          # trials Optuna por modelo
CV_FOLDS     = 5
SCORING      = "neg_root_mean_squared_error"
DIRECTION    = "minimize"   # minimizar RMSE CV

# ── Paths (Drive) ─────────────────────────────────────────────────────────────
BASE = "/content/drive/MyDrive/Mestrado/Aprendizagem Automática/projeto final/experiment"
MODELS_DIR  = os.path.join(BASE, "models_bayes/")
FIGURES_DIR = os.path.join(BASE, "figures_bayes/")
RESULTS_DIR = os.path.join(BASE, "results_bayes/")
MLFLOW_DIR  = os.path.join(BASE, "mlruns_bayes/")
for d in [MODELS_DIR, FIGURES_DIR, RESULTS_DIR]: os.makedirs(d, exist_ok=True)

print(f"GPU        : {GPU_AVAILABLE}")
print(f"Optuna     : {optuna.__version__}")
print(f"N_TRIALS   : {N_TRIALS}")
print(f"Artefatos  : {BASE}")

---
## 3. Configuração do MLflow

In [ ]:
EXPERIMENT_NAME = "regressao-potencia-bayesiana"
mlflow.set_tracking_uri(f"file://{MLFLOW_DIR}")
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"MLflow pronto: {MLFLOW_DIR}")

---
## 4. Carregamento e Pré-processamento

In [ ]:
DATA_PATH = "/content/drive/MyDrive/Mestrado/Aprendizagem Automática/projeto final/articles/vagner_proposal/dataresult_en.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["DATETIME"])
df = df.sort_values("DATETIME").reset_index(drop=True)

TARGET   = "POWERFROMPANELS"
FEATURES = ["IRRADIATION", "ENVTEMPSTATION", "WINDSPEED", "ENVTEMPPANELS"]

print(f"Shape: {df.shape} | Nulos: {df.isnull().sum().sum()}")

In [ ]:
# Features temporais
df["HOUR"]      = df["DATETIME"].dt.hour
df["MINUTE"]    = df["DATETIME"].dt.minute
df["MONTH"]     = df["DATETIME"].dt.month
df["DOY"]       = df["DATETIME"].dt.dayofyear
df["HOUR_SIN"]  = np.sin(2 * np.pi * df["HOUR"]  / 24)
df["HOUR_COS"]  = np.cos(2 * np.pi * df["HOUR"]  / 24)
df["MONTH_SIN"] = np.sin(2 * np.pi * df["MONTH"] / 12)
df["MONTH_COS"] = np.cos(2 * np.pi * df["MONTH"] / 12)
df["IRRAD_TEMP"]= df["IRRADIATION"] * df["ENVTEMPPANELS"]

ALL_FEATURES = FEATURES + [
    "HOUR","MINUTE","MONTH","DOY",
    "HOUR_SIN","HOUR_COS","MONTH_SIN","MONTH_COS","IRRAD_TEMP"
]
print(f"Features: {len(ALL_FEATURES)} → {ALL_FEATURES}")

In [ ]:
# Split ANTES de qualquer limpeza (sem leakage)
split_idx    = int(len(df) * 0.8)
df_train_raw = df.iloc[:split_idx].copy()
df_test_raw  = df.iloc[split_idx:].copy()

# Outliers: Z-score > 4 apenas no treino
z_train  = np.abs(stats.zscore(df_train_raw[TARGET]))
df_train = df_train_raw[z_train < 4].reset_index(drop=True)
df_test  = df_test_raw.reset_index(drop=True)

X_train = df_train[ALL_FEATURES].values
y_train = df_train[TARGET].values
X_test  = df_test[ALL_FEATURES].values
y_test  = df_test[TARGET].values

# Scaler fit apenas no treino
scaler     = RobustScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

tscv = TimeSeriesSplit(n_splits=CV_FOLDS)

print(f"Treino: {X_train.shape} | Teste: {X_test.shape}")
print(f"Período treino : {df_train.DATETIME.iloc[0].date()} → {df_train.DATETIME.iloc[-1].date()}")
print(f"Período teste  : {df_test.DATETIME.iloc[0].date()}  → {df_test.DATETIME.iloc[-1].date()}")

---
## 5. Funções Auxiliares

In [ ]:
def log(msg):
    print(msg, flush=True); sys.stdout.flush()

def evaluate(name, model, X_te, y_te):
    """Avalia modelo já treinado — sem re-treino."""
    y_pred = np.clip(model.predict(X_te), 0, None)
    return {
        "Modelo": name,
        "RMSE"  : np.sqrt(mean_squared_error(y_te, y_pred)),
        "MAE"   : mean_absolute_error(y_te, y_pred),
        "R²"    : r2_score(y_te, y_pred),
        "MAPE%" : np.mean(np.abs((y_te - y_pred) / (y_te + 1e-6))) * 100,
        "y_pred": y_pred,
    }

def cv_rmse(model, X, y):
    """RMSE médio no TimeSeriesSplit — usado como objetivo do Optuna."""
    scores = cross_val_score(model, X, y, cv=tscv,
                             scoring=SCORING, n_jobs=-1 if not GPU_AVAILABLE else 1)
    return -scores.mean()   # retorna RMSE positivo

def run_optuna(name, objective_fn, n_trials=N_TRIALS):
    """
    Executa estudo Optuna com TPE sampler.
    objective_fn(trial) → float (RMSE CV a minimizar)
    Retorna: best_model, best_params, best_cv_rmse, test_res
    """
    safe = name.replace(" ", "_")
    log(f"\n{'='*55}")
    log(f"  Modelo  : {name}")
    log(f"  Sampler : TPE (Optuna {optuna.__version__})")
    log(f"  Trials  : {n_trials}  |  CV: TimeSeriesSplit({CV_FOLDS})")
    log(f"{'='*55}")

    study = optuna.create_study(
        direction=DIRECTION,
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=10),
    )
    study.optimize(objective_fn, n_trials=n_trials, show_progress_bar=False)

    best_params = study.best_params
    best_cv     = study.best_value
    log(f"\n  ✔ {name} — melhor trial #{study.best_trial.number}")
    log(f"  ┌───────────────────────────────────────")
    log(f"  │  CV RMSE : {best_cv:>10,.2f} W")

    # Reconstrói e treina o melhor modelo no treino completo
    best_model = objective_fn(study.best_trial, final=True)
    best_model.fit(X_train_sc, y_train)

    res = evaluate(name, best_model, X_test_sc, y_test)
    log(f"  │  RMSE    : {res['RMSE']:>10,.2f} W")
    log(f"  │  MAE     : {res['MAE']:>10,.2f} W")
    log(f"  │  R²      : {res['R²']:>10.4f}")
    log(f"  │  MAPE    : {res['MAPE%']:>10.2f} %")
    log(f"  └───────────────────────────────────────")
    log(f"  Params: {best_params}")

    # Salva top 10 trials
    df_trials = study.trials_dataframe()
    top10_path = os.path.join(RESULTS_DIR, f"top10_{safe}.csv")
    df_trials.sort_values("value").head(10).to_csv(top10_path, index=False)

    # Salva modelo
    cache_path = os.path.join(MODELS_DIR, f"{safe}.joblib")
    joblib.dump(best_model, cache_path)

    # MLflow
    with mlflow.start_run(run_name=name):
        mlflow.set_tags({"sampler":"TPE","n_trials":n_trials,
                         "cv":f"TimeSeriesSplit({CV_FOLDS})","dataset":"estacao_meteorologica"})
        mlflow.log_params({k: str(v) for k, v in best_params.items()})
        mlflow.log_metrics({"cv_rmse":best_cv,"rmse":res["RMSE"],
                            "mae":res["MAE"],"r2":res["R²"],"mape_pct":res["MAPE%"]})
        mlflow.sklearn.log_model(best_model, "model")
        mlflow.log_artifact(top10_path, "search_results")
        scaler_path = os.path.join(MODELS_DIR, "scaler.joblib")
        joblib.dump(scaler, scaler_path)
        mlflow.log_artifact(scaler_path, "preprocessing")

    return best_model, best_params, best_cv, res

results     = []
best_models = {}
log("Funções definidas.")

---
## 6. Random Forest Regressor

In [ ]:
def rf_objective(trial, final=False):
    params = {
        "n_estimators"     : trial.suggest_int("n_estimators", 100, 700, step=100),
        "max_depth"        : trial.suggest_categorical("max_depth", [None, 8, 10, 12, 15, 20]),
        "min_samples_leaf" : trial.suggest_int("min_samples_leaf", 1, 4),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 8),
        "max_features"     : trial.suggest_categorical("max_features", ["sqrt","log2",3,4,5]),
        "bootstrap"        : trial.suggest_categorical("bootstrap", [True, False]),
        "max_samples"      : trial.suggest_categorical("max_samples", [0.7,0.8,0.9,None]),
    }
    model = RandomForestRegressor(**params, random_state=RANDOM_STATE, n_jobs=-1)
    if final: return model
    return cv_rmse(model, X_train_sc, y_train)

rf_best, rf_params, rf_cv_rmse_val, rf_res = run_optuna("Random Forest", rf_objective)
rf_res["CV_RMSE"] = rf_cv_rmse_val
results.append(rf_res)
best_models["Random Forest"] = rf_best

---
## 7. XGBoost Regressor

In [ ]:
def xgb_objective(trial, final=False):
    params = {
        "n_estimators"    : trial.suggest_int("n_estimators", 100, 1000, step=100),
        "learning_rate"   : trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "max_depth"       : trial.suggest_int("max_depth", 4, 8),
        "subsample"       : trial.suggest_float("subsample", 0.7, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma"           : trial.suggest_float("gamma", 0.0, 0.5),
        "reg_alpha"       : trial.suggest_float("reg_alpha", 0.0, 0.5),
        "reg_lambda"      : trial.suggest_float("reg_lambda", 0.5, 5.0),
    }
    model = XGBRegressor(**params, random_state=RANDOM_STATE,
                         device=XGB_DEVICE,
                         n_jobs=-1 if not GPU_AVAILABLE else 1,
                         verbosity=0)
    if final: return model
    return cv_rmse(model, X_train_sc, y_train)

xgb_best, xgb_params, xgb_cv_rmse_val, xgb_res = run_optuna("XGBoost", xgb_objective)
xgb_res["CV_RMSE"] = xgb_cv_rmse_val
results.append(xgb_res)
best_models["XGBoost"] = xgb_best

---
## 8. LightGBM Regressor

In [ ]:
def lgbm_objective(trial, final=False):
    params = {
        "n_estimators"     : trial.suggest_int("n_estimators", 200, 1000, step=100),
        "learning_rate"    : trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves"       : trial.suggest_int("num_leaves", 31, 100),
        "max_depth"        : trial.suggest_categorical("max_depth", [-1,5,6,7,8,10]),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "subsample"        : trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree" : trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "reg_alpha"        : trial.suggest_float("reg_alpha", 0.0, 0.5),
        "reg_lambda"       : trial.suggest_float("reg_lambda", 0.0, 1.0),
    }
    model = LGBMRegressor(**params, random_state=RANDOM_STATE,
                          device=LGBM_DEVICE,
                          n_jobs=-1 if not GPU_AVAILABLE else 1,
                          verbose=-1)
    if final: return model
    return cv_rmse(model, X_train_sc, y_train)

lgbm_best, lgbm_params, lgbm_cv_rmse_val, lgbm_res = run_optuna("LightGBM", lgbm_objective)
lgbm_res["CV_RMSE"] = lgbm_cv_rmse_val
results.append(lgbm_res)
best_models["LightGBM"] = lgbm_best

---
## 9. CatBoost Regressor

In [ ]:
def cat_objective(trial, final=False):
    params = {
        "iterations"         : trial.suggest_int("iterations", 200, 1000, step=100),
        "learning_rate"      : trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "depth"              : trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg"        : trial.suggest_float("l2_leaf_reg", 1.0, 7.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 2.0),
        "random_strength"    : trial.suggest_float("random_strength", 0.0, 2.0),
        "border_count"       : trial.suggest_categorical("border_count", [32, 64, 128]),
    }
    model = CatBoostRegressor(**params, random_state=RANDOM_STATE,
                              task_type=CAT_TASK_TYPE, verbose=0)
    if final: return model
    return cv_rmse(model, X_train_sc, y_train)

cat_best, cat_params, cat_cv_rmse_val, cat_res = run_optuna("CatBoost", cat_objective)
cat_res["CV_RMSE"] = cat_cv_rmse_val
results.append(cat_res)
best_models["CatBoost"] = cat_best

---
## 10. KNN Regressor

In [ ]:
def knn_objective(trial, final=False):
    params = {
        "n_neighbors": trial.suggest_int("n_neighbors", 3, 25),
        "weights"    : trial.suggest_categorical("weights", ["uniform","distance"]),
        "metric"     : trial.suggest_categorical("metric", ["euclidean","manhattan","minkowski"]),
        "p"          : trial.suggest_categorical("p", [1, 2]),
        "leaf_size"  : trial.suggest_int("leaf_size", 20, 40),
        "algorithm"  : trial.suggest_categorical("algorithm", ["auto","ball_tree","kd_tree"]),
    }
    model = KNeighborsRegressor(**params, n_jobs=-1)
    if final: return model
    return cv_rmse(model, X_train_sc, y_train)

knn_best, knn_params, knn_cv_rmse_val, knn_res = run_optuna("KNN", knn_objective)
knn_res["CV_RMSE"] = knn_cv_rmse_val
results.append(knn_res)
best_models["KNN"] = knn_best

---
## 11. SARIMA Regression

> SARIMA é um modelo de série temporal puro — baseline temporal.
> `auto_arima` já faz seleção automática de ordem via AIC (equivalente a otimização bayesiana stepwise).

In [ ]:
df_daily_train = df_train.set_index("DATETIME")[TARGET].resample("D").sum().dropna()
df_daily_test  = df_test.set_index("DATETIME")[TARGET].resample("D").sum().dropna()
df_daily       = pd.concat([df_daily_train, df_daily_test])
train_s, test_s = df_daily_train, df_daily_test

print(f"Série diária — treino: {len(train_s)} dias | teste: {len(test_s)} dias")

sarima_model = auto_arima(
    train_s, seasonal=True, m=7,
    start_p=0, max_p=3, start_q=0, max_q=3,
    start_P=0, max_P=2, start_Q=0, max_Q=2,
    d=None, D=None, information_criterion="aic",
    stepwise=True, error_action="ignore",
    suppress_warnings=True, n_fits=50
)
print(sarima_model.summary())

In [ ]:
sarima_pred = np.clip(sarima_model.predict(n_periods=len(test_s)), 0, None)
rmse_s = np.sqrt(mean_squared_error(test_s, sarima_pred))
mae_s  = mean_absolute_error(test_s, sarima_pred)
r2_s   = r2_score(test_s, sarima_pred)
mape_s = np.mean(np.abs((test_s.values - sarima_pred) / (test_s.values + 1e-6))) * 100

log(f"SARIMA → RMSE={rmse_s:,.1f}  MAE={mae_s:,.1f}  R²={r2_s:.4f}  MAPE%={mape_s:.2f}")
log("(métricas em Wh diário — escala diferente dos modelos por minuto)")

with mlflow.start_run(run_name="SARIMA"):
    mlflow.set_tags({"model":"SARIMA","granularity":"daily"})
    mlflow.log_params({"order":str(sarima_model.order),
                       "seasonal_order":str(sarima_model.seasonal_order),"m":7})
    mlflow.log_metrics({"rmse":rmse_s,"mae":mae_s,"r2":r2_s,"mape_pct":mape_s})
    sarima_path = os.path.join(MODELS_DIR, "SARIMA.joblib")
    joblib.dump(sarima_model, sarima_path)
    mlflow.log_artifact(sarima_path, "model")

results.append({"Modelo":"SARIMA","RMSE":rmse_s,"MAE":mae_s,
                "R²":r2_s,"MAPE%":mape_s,"CV_RMSE":float("nan"),
                "y_pred":sarima_pred})

---
## 12. Comparação de Resultados

In [ ]:
res_df = pd.DataFrame([{k:v for k,v in r.items() if k != "y_pred"}
                        for r in results]).set_index("Modelo")
res_df_sorted = res_df.sort_values("RMSE")
print(res_df_sorted.round(2).to_string())

In [ ]:
supervised = res_df.drop("SARIMA", errors="ignore")
metrics    = ["RMSE","MAE","R²","MAPE%"]
pal = sns.color_palette(PALETTE, len(supervised))

fig, axes = plt.subplots(1, 4, figsize=(FIG_W*1.1, FIG_H), dpi=DPI)
fig.suptitle("Comparação dos Modelos — Bayesiana (Optuna TPE)")

for ax, metric in zip(axes, metrics):
    data = supervised[metric].sort_values(ascending=(metric != "R²"))
    bars = ax.barh(data.index, data.values, color=pal[:len(data)],
                   edgecolor="white", linewidth=0.5)
    ax.set_title(metric); ax.set_xlabel(metric)
    for bar, val in zip(bars, data.values):
        ax.text(val*1.01, bar.get_y()+bar.get_height()/2,
                f"{val:.2f}", va="center", fontsize=8)

plt.tight_layout(); plt.show()

---
## 13. Predito vs Real — Melhor Modelo

In [ ]:
best_name = supervised["RMSE"].idxmin()
best_pred = next(r["y_pred"] for r in results if r["Modelo"] == best_name)
log(f"Melhor modelo: {best_name}")

fig, axes = plt.subplots(1, 2, figsize=(FIG_W, FIG_H+0.5), dpi=DPI)
fig.suptitle(f"{best_name} — Predito vs Real (Optuna TPE)")

ax = axes[0]
ax.scatter(y_test, best_pred, alpha=0.15, s=6, color=PRIMARY, linewidths=0)
lim = max(y_test.max(), best_pred.max())
ax.plot([0,lim],[0,lim], color=ACCENT, linewidth=1.5, linestyle="--", label="Ideal")
ax.set_xlabel("Real (W)"); ax.set_ylabel("Predito (W)")
ax.set_title("Scatter: Real vs Predito"); ax.legend()

ax = axes[1]
ax.plot(y_test[:500],  color=NEUTRAL, linewidth=0.8, label="Real",    alpha=0.8)
ax.plot(best_pred[:500], color=ACCENT, linewidth=0.8, label="Predito", alpha=0.9)
ax.set_xlabel("Amostra"); ax.set_ylabel("Potência (W)")
ax.set_title("Série Temporal (500 pts)"); ax.legend()

plt.tight_layout(); plt.show()

In [ ]:
residuals = y_test - best_pred
fig, axes = plt.subplots(1, 3, figsize=(FIG_W, FIG_H), dpi=DPI)
fig.suptitle(f"{best_name} — Análise de Resíduos")

sns.histplot(residuals, bins=60, kde=True, color=PRIMARY, ax=axes[0])
axes[0].axvline(0, color=ACCENT, linestyle="--", linewidth=1.5)
axes[0].set_title("Distribuição"); axes[0].set_xlabel("Resíduo (W)")

axes[1].scatter(best_pred, residuals, alpha=0.15, s=5, color=PRIMARY, linewidths=0)
axes[1].axhline(0, color=ACCENT, linestyle="--", linewidth=1.5)
axes[1].set_xlabel("Predito (W)"); axes[1].set_ylabel("Resíduo (W)")
axes[1].set_title("Resíduo vs Predito")

stats.probplot(residuals, dist="norm", plot=axes[2])
axes[2].set_title("Q-Q Plot")

plt.tight_layout(); plt.show()

---
## 14. Importância de Features — Melhor Modelo

In [ ]:
best_obj = best_models[best_name]
if hasattr(best_obj, "feature_importances_"):
    imp = pd.Series(best_obj.feature_importances_, index=ALL_FEATURES).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(9,5), dpi=DPI)
    ax.barh(imp.index, imp.values,
            color=[PRIMARY if v==imp.max() else NEUTRAL for v in imp.values],
            edgecolor="white")
    ax.set_title(f"Importância de Features — {best_name}")
    ax.set_xlabel("Importância")
    plt.tight_layout(); plt.show()
else:
    from sklearn.inspection import permutation_importance
    perm = permutation_importance(best_obj, X_test_sc, y_test,
                                  n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
    imp = pd.Series(perm.importances_mean, index=ALL_FEATURES).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(9,5), dpi=DPI)
    ax.barh(imp.index, imp.values, color=PRIMARY, edgecolor="white")
    ax.set_title(f"Permutation Importance — {best_name}")
    ax.set_xlabel("Redução média no RMSE")
    plt.tight_layout(); plt.show()

---
## 15. Curva de Otimização — Histórico de Trials

Mostra como o Optuna convergiu ao longo dos trials para cada modelo.

In [ ]:
# Recria os estudos para plotar a convergência
# (requer reexecutar o notebook com study salvo — aqui usamos os resultados do res_df)

fig, ax = plt.subplots(figsize=(FIG_W*0.8, FIG_H), dpi=DPI)

model_names = [r["Modelo"] for r in results if r["Modelo"] != "SARIMA"]
colors_conv = sns.color_palette(PALETTE, len(model_names))

for name, color in zip(model_names, colors_conv):
    top10_path = os.path.join(RESULTS_DIR, f"top10_{name.replace(" ","_")}.csv")
    if os.path.exists(top10_path):
        df_t = pd.read_csv(top10_path).sort_values("value")
        ax.plot(range(1, len(df_t)+1), df_t["value"].values,
                marker="o", markersize=4, label=name, color=color, linewidth=1.5)

ax.set_xlabel("Trial rank (top 10)")
ax.set_ylabel("CV RMSE (W)")
ax.set_title("Top 10 Trials por Modelo — Optuna TPE")
ax.legend()
plt.tight_layout(); plt.show()

---
## 16. SARIMA — Visualização da Predição Diária

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(FIG_W, FIG_H*2), dpi=DPI)
fig.suptitle("SARIMA — Geração Diária Total (Wh)")

axes[0].plot(df_daily.index, df_daily.values, color=NEUTRAL, linewidth=1, label="Real")
axes[0].plot(test_s.index, sarima_pred, color=ACCENT, linewidth=1.5, label="SARIMA Predito")
axes[0].axvline(test_s.index[0], color="red", linestyle="--", linewidth=1, alpha=0.7, label="Início teste")
axes[0].set_title("Série completa + Predição"); axes[0].set_ylabel("Wh/dia"); axes[0].legend()

axes[1].plot(test_s.index, test_s.values, color=NEUTRAL, linewidth=1.2, label="Real")
axes[1].plot(test_s.index, sarima_pred, color=ACCENT, linewidth=1.5, label="SARIMA Predito")
axes[1].fill_between(test_s.index, test_s.values, sarima_pred, alpha=0.2, color="red", label="Erro")
axes[1].set_title(f"Conjunto de Teste  |  RMSE={rmse_s:,.0f} Wh  R²={r2_s:.3f}")
axes[1].set_ylabel("Wh/dia"); axes[1].legend()

plt.tight_layout(); plt.show()

---
## 17. Melhores Hiperparâmetros Encontrados

In [ ]:
best_params_all = {
    "Random Forest": rf_params,
    "XGBoost"      : xgb_params,
    "LightGBM"     : lgbm_params,
    "CatBoost"     : cat_params,
    "KNN"          : knn_params,
    "SARIMA"       : {"order": sarima_model.order,
                      "seasonal_order": sarima_model.seasonal_order},
}
print("="*60)
print("MELHORES HIPERPARÂMETROS — Optuna TPE")
print("="*60)
for name, params in best_params_all.items():
    print(f"\n[{name}]")
    for k, v in params.items():
        print(f"  {k}: {v}")

---
## 18. Resumo Final

In [ ]:
print("="*60)
print("RESUMO FINAL — Otimização Bayesiana (Optuna TPE)")
print("="*60)
print(f"\nDataset treino : {len(df_train):,} | Teste: {len(df_test):,}")
print(f"Features       : {len(ALL_FEATURES)}")
print(f"Sampler        : TPE | N_TRIALS: {N_TRIALS} | CV: TimeSeriesSplit({CV_FOLDS})")
print(f"Scaler         : RobustScaler | Leakage: Nenhum")
print()
print("Ranking por RMSE (conjunto de teste):")
print(res_df_sorted[["RMSE","MAE","R²","MAPE%"]].round(2).to_string())
print(f"\nMelhor modelo  : {best_name}")